In [1]:
from datasets import load_dataset
import pandas as pd
import re


In [2]:
ds_raw  = load_dataset("MahsaVafaie/BZKopen", "raw")
ds_norm = load_dataset("MahsaVafaie/BZKopen", "normalized")

DATE_COLS = ["ApplicantBirthDate", "VictimBirthDate", "VictimDeathDate"]

def hf_to_df(ds):
    dfs = []
    for split in ["train", "validation", "test"]:
        df = ds[split].to_pandas().drop(columns=["image"])
        df["split"] = split
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

raw_df  = hf_to_df(ds_raw)
norm_df = hf_to_df(ds_norm)

print(f"Total rows: {len(raw_df)}")
print(f"Splits:     {raw_df['split'].value_counts().to_dict()}")
raw_df[["split"] + DATE_COLS].head()


Total rows: 516
Splits:     {'train': 361, 'test': 78, 'validation': 77}


,split,ApplicantBirthDate,VictimBirthDate,VictimDeathDate
0,train,17.4.1819,,
1,train,6.6.1819,,
2,train,,25.8.1829,
3,train,,13.3.50,
4,train,,8.7.50,


In [91]:
pairs = pd.concat([
    raw_df[DATE_COLS].rename(columns=lambda c: c + "_raw"),
    norm_df[DATE_COLS].rename(columns=lambda c: c + "_norm"),
], axis=1)
mask = raw_df[DATE_COLS].apply(lambda col: col.str.len() > 0).any(axis=1)

In [ ]:
GERMAN_MONTHS = {
    'januar': 1,  'jan': 1,
    'februar': 2, 'feb': 2,
    'maerz': 3,   'märz': 3, 'mar': 3, 'mär': 3,
    'april': 4,   'apr': 4,
    'mai': 5,
    'juni': 6,    'jun': 6,
    'juli': 7,    'jul': 7,
    'august': 8,  'aug': 8,
    'september': 9, 'sep': 9, 'sept': 9,
    'oktober': 10, 'okt': 10, 'oct': 10,
    'november': 11, 'nov': 11,
    'dezember': 12, 'dez': 12, 'dec': 12,
}

ROMAN = {
    'i': 1, 'ii': 2, 'iii': 3, 'iv': 4, 'v': 5, 'vi': 6,
    'vii': 7, 'viii': 8, 'ix': 9, 'x': 10, 'xi': 11, 'xii': 12,
}

NON_DATE_TOKENS = {
    'unbekannt', 'deportiert', 'verstorben', 'verst', 'deportation',
    'umgekommen', 'gestorben', 'beide', 'unk',
}

SEASON_TO_MONTH = {'sommer': 8, 'frühling': 4, 'herbst': 10, 'winter': 1}

_DATE_TOKEN = re.compile(
    r'\b(\d{1,2})[.\-/]+\s*(\d{1,2})[.\-/]+\s*(\d{2,4})\b'
)

MANUAL = 'MANUAL'  # flag for 2-digit years that require manual completion


def _century(yy, col):
    """Resolve a 2-digit year to a full year based on col rules.

    Rules (birth/death cannot be later than 1954):

    ApplicantBirthDate:
      55-99 -> 18xx  |  00-33 -> 19xx  |  34-54 -> MANUAL

    VictimBirthDate:
      45-99 -> 18xx  |  00-33 -> 19xx  |  34-44 -> MANUAL

    Any death date (VictimDeathDate):
      always 19xx

    Fallback (unknown col): >= 50 -> 18xx, < 50 -> 19xx
    """
    if 'DeathDate' in (col or ''):
        return 1900
    if col == 'ApplicantBirthDate':
        if yy >= 55:
            return 1800
        if yy <= 33:
            return 1900
        return MANUAL          # 34-54
    if col == 'VictimBirthDate':
        if yy >= 45:
            return 1800
        if yy <= 33:
            return 1900
        return MANUAL          # 34-44
    # fallback
    return 1800 if yy >= 50 else 1900


def _fix_component(v):
    """Reverse digits of an out-of-range day/month (e.g. 40 -> 04)."""
    if v > 31 and v >= 10:
        rev = int(str(v)[::-1])
        if 1 <= rev <= 31:
            return rev
    return v


def _to_iso(day, month, year):
    try:
        d, m, y = _fix_component(int(day)), int(month), int(year)
        m = _fix_component(m)
        if m > 12 and d <= 12:
            d, m = m, d
        if not (1 <= m <= 12 and 1 <= d <= 31):
            return ''
        return f'{y:04d}-{m:02d}-{d:02d}'
    except (ValueError, TypeError):
        return ''


def _find_all_date_tokens(s):
    return [(h.group(1), h.group(2), h.group(3)) for h in _DATE_TOKEN.finditer(s)]


def _resolve_year(yr_str, col):
    """Return full year int, or MANUAL if ambiguous 2-digit year needs human review."""
    yr_int = int(yr_str)
    if yr_int >= 100:
        return yr_int
    century = _century(yr_int, col)
    if century is MANUAL:
        return MANUAL
    return century + yr_int


def _normalize_single(s, col=None):
    """Normalize one clean date string to ISO or '' or MANUAL."""
    s = s.strip()
    s = re.sub(r'\s*\([^)]*\)', '', s).strip()
    s = re.sub(r'\.$', '', s).strip()
    if not s:
        return ''

    hit = re.match(r'^(\d{4})-(\d{2})-(\d{2})', s)
    if hit:
        return f'{hit.group(1)}-{hit.group(2)}-{hit.group(3)}'

    lower = s.lower()

    if any(tok in lower for tok in NON_DATE_TOKENS):
        tokens = _find_all_date_tokens(s)
        if tokens:
            d, mo, yr = tokens[0]
            resolved = _resolve_year(yr, col)
            if resolved is MANUAL:
                return MANUAL
            return _to_iso(d, mo, resolved)
        month_map = {**GERMAN_MONTHS, **SEASON_TO_MONTH}
        for mon_str, mo in month_map.items():
            if mon_str in lower:
                hit2 = re.search(r'\b(\d{4})\b', s)
                if hit2:
                    return f'{hit2.group(1)}-{mo:02d}-01'
        return ''

    if s == '?':
        return ''

    hit = re.match(r'^\?\.\s*(\d{4})$', s)
    if hit:
        return f'{hit.group(1)}-01-01'

    # D.M.YYYY or D.M.YY — permissive separators
    hit = re.match(r'^(\d{1,2})[.\-/]+\s*(\d{1,2})[.\-/]+\s*(\d{2,4})$', s)
    if hit:
        resolved = _resolve_year(hit.group(3), col)
        if resolved is MANUAL:
            return MANUAL
        return _to_iso(hit.group(1), hit.group(2), resolved)

    # MM/DD/YYYY US
    hit = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', s)
    if hit:
        return _to_iso(hit.group(2), hit.group(1), hit.group(3))

    # D Roman/alpha YYYY: "3.II.1887", "13. Aug. 1865"
    hit = re.match(r'^(\d{1,2})[.\s]+([A-Za-z\u00c4\u00e4\u00d6\u00f6\u00dc\u00fc\u00df]+)\.?\s*(\d{2,4})$', s)
    if hit:
        mon_str = hit.group(2).lower().rstrip('.')
        mo = ROMAN.get(mon_str) or GERMAN_MONTHS.get(mon_str)
        if mo:
            resolved = _resolve_year(hit.group(3), col)
            if resolved is MANUAL:
                return MANUAL
            return _to_iso(hit.group(1), mo, resolved)

    # "Month YYYY"
    hit = re.match(r'^([A-Za-z\u00c4\u00e4\u00d6\u00f6\u00dc\u00fc\u00df]+)\.?\s*(\d{4})$', s)
    if hit:
        mon_str = hit.group(1).lower().rstrip('.')
        mo = GERMAN_MONTHS.get(mon_str) or SEASON_TO_MONTH.get(mon_str)
        if mo:
            return f'{hit.group(2)}-{mo:02d}-01'

    # "Ende YYYY"
    hit = re.match(r'^[Ee]nde\s+(\d{4})$', s)
    if hit:
        return f'{hit.group(1)}-12-01'

    # YYYY only
    hit = re.match(r'^(\d{4})$', s)
    if hit:
        return f'{hit.group(1)}-01-01'

    # M.YYYY: "2.1906"
    hit = re.match(r'^(\d{1,2})\.(\d{4})$', s)
    if hit:
        return f'{hit.group(2)}-{int(hit.group(1)):02d}-01'


    tokens = _find_all_date_tokens(s)
    if tokens:
        d, mo, yr = tokens[0]
        resolved = _resolve_year(yr, col)
        if resolved is MANUAL:
            return MANUAL
        return _to_iso(d, mo, resolved)

    return ''


_MULTI_LABELED = re.compile(r'[ab1-9]\)\s*')
_TWO_DATES = re.compile(
    r'^(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})\s+(\d{1,2}[.\-/]+\d{1,2}[.\-/]+\d{2,4})$'
)


def normalize_date(raw, col=None):
    """Normalise raw date string to ISO 8601 or MANUAL. Multiple dates -> 'D1 ; D2'."""
    if not raw or not raw.strip():
        return ''
    s = raw.strip()
    s_clean = re.sub(r'\s*\([^)]*\)', '', s).strip()

    parts = [p.strip() for p in _MULTI_LABELED.split(s_clean) if p.strip()]
    if len(parts) > 1:
        results = [r for r in [_normalize_single(p, col) for p in parts] if r]
        return ' ; '.join(results) if results else ''

    if ';' in s_clean:
        parts = [p.strip() for p in s_clean.split(';') if p.strip()]
        results = [r for r in [_normalize_single(p, col) for p in parts] if r]
        return ' ; '.join(results) if results else ''

    hit = _TWO_DATES.match(s_clean)
    if hit:
        results = [r for r in [_normalize_single(hit.group(1), col), _normalize_single(hit.group(2), col)] if r]
        return ' ; '.join(results) if results else ''

    return _normalize_single(s_clean, col)


In [70]:
def eval_normalization(split='validation'):
    mask = raw_df['split'] == split
    raw_split  = raw_df[mask].reset_index(drop=True)
    norm_split = norm_df[mask].reset_index(drop=True)

    rows = []
    for i in range(len(raw_split)):
        for col in DATE_COLS:
            raw_val = raw_split.at[i, col]
            gt_val  = norm_split.at[i, col]
            if not raw_val:
                continue
            pred_val = normalize_date(raw_val, col=col)
            rows.append({
                'split': split,
                'record_idx': i,
                'col': col, 'raw': raw_val,
                'pred': pred_val, 'gt': gt_val,
                'correct': pred_val == gt_val,
            })
    df = pd.DataFrame(rows)
    n          = len(df)
    n_correct  = df['correct'].sum()
    n_manual   = (df['pred'] == 'MANUAL').sum()
    has_gt     = df['gt'].notna() & (df['gt'] != '')
    n_gt       = has_gt.sum()
    n_correct_gt = df.loc[has_gt, 'correct'].sum()
    print(
        f"[{split}]  all: {n_correct}/{n} ({100*n_correct/n:.1f}%)"
        f"  |  with GT: {n_correct_gt}/{n_gt} ({100*n_correct_gt/n_gt:.1f}%)"
        f"  |  MANUAL: {n_manual}"
    )
    return df

val_results   = eval_normalization('validation')
train_results = eval_normalization('train')
test_results  = eval_normalization('test')


[validation]  all: 83/91 (91.2%)  |  with GT: 82/84 (97.6%)  |  MANUAL: 0
[train]  all: 443/469 (94.5%)  |  with GT: 430/434 (99.1%)  |  MANUAL: 0
[test]  all: 99/106 (93.4%)  |  with GT: 93/94 (98.9%)  |  MANUAL: 0


In [90]:
errors = test_results[~test_results["correct"]].copy()
print(f"Errors on test: {len(errors)}")
errors[["record_idx", "raw", "pred", "gt", "col"]]


Errors on test: 7


,record_idx,raw,pred,gt,col
8,4,1944,1944-01-01,,VictimDeathDate
24,13,24.6.1941,1941-06-24,,VictimDeathDate
26,15,1.1.76,1876-01-01,1876-01-04,VictimBirthDate
36,22,3.8.1944,1944-08-03,,VictimDeathDate
38,23,25.8.1940,1940-08-25,,VictimDeathDate
70,51,26. März 1903,1903-03-26,,ApplicantBirthDate
105,77,6.3.1941,1941-03-06,,VictimDeathDate


In [89]:
DISPLAY_COLS = [
   'ApplicantBirthDate', 'VictimBirthDate', 'VictimDeathDate'
]


def show_record(i, split='test'):
    """Show all fields of record i (position within split) as a dataframe row."""
    mask     = raw_df['split'] == split
    raw_row  = raw_df[mask].reset_index(drop=True).iloc[i]
    norm_row = norm_df[mask].reset_index(drop=True).iloc[i]

    rows = []
    for col in DISPLAY_COLS:
        raw_val  = raw_row.get(col, '')
        norm_val = norm_row.get(col, '') if col in DATE_COLS else ''
        if col in DATE_COLS:
            pred_val = normalize_date(raw_val, col=col) if raw_val else ''
            rows.append({'field': col, 'raw': raw_val, 'pred': pred_val, 'gt': norm_val})
        else:
            rows.append({'field': col, 'raw': raw_val, 'pred': '', 'gt': '', 'status': ''})

    return pd.DataFrame(rows).set_index('field')

show_record(4)


,raw,pred,gt
field,,,
ApplicantBirthDate,24.4.1858,1858-04-24,1858-04-24
VictimBirthDate,5.6.08,1908-06-05,1908-06-05
VictimDeathDate,1944,1944-01-01,
